# 07 — Ringkasan AI Grounded
**SuaraLens** | Generate ringkasan naratif otomatis dari angka agregat menggunakan LLM.
Input LLM = angka saja (bukan teks mentah) untuk meminimalkan risiko halusinasi.


In [ ]:
import sys
sys.path.insert(0, '..')

import json
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
from pathlib import Path
from modules.analytics  import build_sla_analytics_payload, build_trend_analytics_payload
from modules.ai_summary import generate_summary, check_grounding, generate_summary_template, save_summary_json
from modules.llm_engine import check_ollama_status

DATA_PATH   = '../data/suaralens_dummy_simulasi.jsonl'
OUTPUT_PATH = '../data/output/ai_summary.json'

df = pd.read_json(DATA_PATH, lines=True)
print(f'Dataset: {len(df):,} baris')
print(f'Ollama status: {check_ollama_status()["running"]}')


## 1. Siapkan Data Agregat dari analytics.py

In [ ]:
sla_data   = build_sla_analytics_payload(df)
trend_data = build_trend_analytics_payload(df, top_n=5)

# Kombinasikan untuk satu ringkasan komprehensif
combined_data = {
    'periode':             trend_data['date_range'],
    'total_masukan':       sla_data['total_records'],
    'overall_breach_pct':  sla_data['overall_breach_pct'],
    'top3_breach_kategori': sla_data['by_category'][:3],
    'top3_kategori':       [
        {'kategori': k, 'jumlah': v}
        for k, v in sorted(
            pd.read_json(DATA_PATH, lines=True)['kategori_true'].value_counts().items(),
            key=lambda x: -x[1]
        )[:3]
    ],
    'avg_resolution_top3': sla_data['avg_resolution_days'][:3],
    'urgency_breach':      sla_data['by_urgency'],
}

print('Data agregat siap:')
print(json.dumps(combined_data, ensure_ascii=False, indent=2, default=str)[:1500])
print('...')


## 2. Generate Ringkasan — Skenario Normal

In [ ]:
result = generate_summary(combined_data, context='laporan bulanan SuaraLens')

print('=== Ringkasan AI ===')
print(result.get('summary', '[GAGAL]'))
print()
print(f'Model   : {result["model_used"]}')
print(f'Waktu   : {result["inference_time_s"]}s')
print(f'Error   : {result["error"]}')


## 3. Uji Grounding

In [ ]:
grounding = check_grounding(result.get('summary', ''), combined_data)

print('=== Grounding Check ===')
print(f'Angka dalam ringkasan     : {grounding["numbers_in_summary"]}')
print(f'Berpotensi tidak grounded : {grounding["potentially_ungrounded"]}')
print(f'Catatan                   : {grounding["grounding_note"]}')


## 4. Uji 4 Skenario Data Berbeda

In [ ]:
scenarios = [
    {
        'label': 'Skenario 1: Breach Rendah',
        'data': {
            'total_masukan': 1200,
            'overall_breach_pct': 8.5,
            'top3_breach_kategori': [
                {'kategori': 'Fasilitas', 'breach_pct': 12.0},
                {'kategori': 'Sarana IT', 'breach_pct': 9.1},
                {'kategori': 'Parkir & Keamanan', 'breach_pct': 7.3},
            ]
        }
    },
    {
        'label': 'Skenario 2: Breach Sangat Tinggi',
        'data': {
            'total_masukan': 3500,
            'overall_breach_pct': 58.2,
            'top3_breach_kategori': [
                {'kategori': 'Keuangan', 'breach_pct': 71.0},
                {'kategori': 'Akademik', 'breach_pct': 65.4},
                {'kategori': 'Fasilitas', 'breach_pct': 52.8},
            ]
        }
    },
    {
        'label': 'Skenario 3: Volume Tinggi tapi Breach Moderat',
        'data': {
            'total_masukan': 8900,
            'overall_breach_pct': 25.0,
            'dominan_kategori': 'Akademik',
            'avg_resolution_days': 7.2,
        }
    },
    {
        'label': 'Skenario 4: Data Lengkap Multiaspek',
        'data': combined_data
    },
]

scenario_results = []
for scenario in scenarios:
    res = generate_summary(scenario['data'], context=scenario['label'])
    gr  = check_grounding(res.get('summary', ''), scenario['data'])
    print(f"=== {scenario['label']} ===")
    print(res.get('summary', '[GAGAL]'))
    print(f"Ungrounded numbers: {gr['potentially_ungrounded']}")
    print()
    scenario_results.append({
        'scenario':  scenario['label'],
        'summary':   res.get('summary'),
        'grounding': gr,
    })


## 5. Skenario Jebakan: Data Minim

In [ ]:
trap_data = {
    'total_masukan': 12,
    'periode': {'start': '2026-08-01', 'end': '2026-08-07'},
    # Sengaja tidak ada informasi breakdown kategori, breach, dll
}

print('=== Skenario Jebakan: Data Sangat Minim ===')
trap_result = generate_summary(trap_data, context='laporan mingguan (data sangat terbatas)')
print('Summary:', trap_result.get('summary', '[GAGAL]'))
print()

trap_grounding = check_grounding(trap_result.get('summary', ''), trap_data)
print('Grounding check:', trap_grounding['grounding_note'])
print()
print('Analisis: Apakah model jujur mengakui data terbatas, atau tetap mengarang kesimpulan?')


## 6. Simpan Hasil ke JSON

In [ ]:
final_output = {
    'generated_at':      pd.Timestamp.now().isoformat(),
    'main_summary':      result,
    'main_grounding':    grounding,
    'scenario_results':  scenario_results,
    'trap_scenario': {
        'summary':   trap_result,
        'grounding': trap_grounding,
    },
}

save_summary_json(final_output, OUTPUT_PATH)
print(f'Semua hasil disimpan ke: {OUTPUT_PATH}')


## 7. Catatan: Strategi Mitigasi jika Grounding Gagal

| Strategi | Kapan digunakan |
|---|---|
| **Template terstruktur** | Jika grounding check menunjukkan banyak angka tidak tertelusuri |
| **Batasi num_predict** | Kurangi panjang output agar model tidak ngelantur |
| **Turunkan temperature** | Temperature < 0.1 untuk output lebih deterministik |
| **Prompt chain** | Generate → Verify → Filter: tambahkan step verifikasi sebelum ditampilkan |
| **Human review gate** | Semua ringkasan AI perlu approval reviewer sebelum tampil di dashboard pimpinan |

> Fungsi `generate_summary_template()` di `modules/ai_summary.py` tersedia sebagai fallback yang 
> lebih aman — angka di-inject langsung ke template string, tanpa risiko halusinasi.
